### Add two vectors element-wise on GPU

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from numba import cuda, float32
import numpy as np

@cuda.jit
def sum_matrix_rowwise_loop(mat, result, rows, cols):
    # Shared memory to hold all matrix elements
    sdata = cuda.shared.array(16, dtype=float32)
    tid = cuda.threadIdx.x

    # Load matrix into shared memory
    if tid < rows * cols:
        sdata[tid] = mat[tid]
    else:
        sdata[tid] = 0.0
    cuda.syncthreads()

    # -------------------------
    # Step 1: Row-wise reduction
    # -------------------------
    # Each iteration halves the number of rows:
    # bottom half adds to top half
    stride_rows = rows // 2
    while stride_rows > 0:
        if tid < stride_rows * cols:
            sdata[tid] += sdata[tid + stride_rows * cols]
        stride_rows //= 2
        cuda.syncthreads()
        print("Thread", tid, "=> sdata[tid] =", sdata[tid])

    # -------------------------
    # Step 2: Column-wise reduction
    # -------------------------
    # Now we have a single row (of 'cols' elements)
    stride_cols = cols // 2
    while stride_cols > 0:
        if tid < stride_cols:
            sdata[tid] += sdata[tid + stride_cols]
        stride_cols //= 2
        cuda.syncthreads()
        print("Thread", tid, "=> sdata[tid] =", sdata[tid])
    # Thread 0 writes final result
    if tid == 0:
        result[0] = sdata[0]


In [3]:
# Host (CPU) code
A = np.arange(16, dtype=np.float32).reshape(4, 4)
A = np.array([[0.73, 0.42, 0.89, 0.15], [0.67, 0.91, 0.28, 0.55], [0.82, 0.36, 0.71, 0.49], [0.93, 0.18, 0.64, 0.27]])
print("Input Matrix A:")
print(A)

# Flatten for GPU
A_flat = A.flatten()
result = np.zeros(1, dtype=np.float32)

threads_per_block = 16
blocks_per_grid = 1

# Launch kernel with matrix shape info
sum_matrix_rowwise_loop[blocks_per_grid, threads_per_block](A_flat, result, 4, 4)

print("\nSum of all elements in matrix A:", result[0])


Input Matrix A:
[[0.73 0.42 0.89 0.15]
 [0.67 0.91 0.28 0.55]
 [0.82 0.36 0.71 0.49]
 [0.93 0.18 0.64 0.27]]
Thread 0 => sdata[tid] = 1.550000
Thread 1 => sdata[tid] = 0.780000
Thread 2 => sdata[tid] = 1.600000
Thread 3 => sdata[tid] = 0.640000
Thread 4 => sdata[tid] = 1.600000
Thread 5 => sdata[tid] = 1.090000
Thread 6 => sdata[tid] = 0.920000
Thread 7 => sdata[tid] = 0.820000
Thread 8 => sdata[tid] = 0.820000
Thread 9 => sdata[tid] = 0.360000
Thread 10 => sdata[tid] = 0.710000
Thread 11 => sdata[tid] = 0.490000
Thread 12 => sdata[tid] = 0.930000
Thread 13 => sdata[tid] = 0.180000
Thread 14 => sdata[tid] = 0.640000
Thread 15 => sdata[tid] = 0.270000
Thread 0 => sdata[tid] = 3.150000
Thread 1 => sdata[tid] = 1.870000
Thread 2 => sdata[tid] = 2.520000
Thread 3 => sdata[tid] = 1.460000
Thread 4 => sdata[tid] = 1.600000
Thread 5 => sdata[tid] = 1.090000
Thread 6 => sdata[tid] = 0.920000
Thread 7 => sdata[tid] = 0.820000
Thread 8 => sdata[tid] = 0.820000
Thread 9 => sdata[tid] = 0.360000
T

In [4]:
from numba import cuda, float32
import numpy as np

# -------------------------------
# CUDA Kernel
# -------------------------------
@cuda.jit
def block_tile_sum_loop(mat, block_sums, rows_per_block, cols_per_block):
    # Shared memory for 4x4 = 16 elements per block
    sdata = cuda.shared.array(16, dtype=float32)
    
    tid = cuda.threadIdx.x
    bx = cuda.blockIdx.x  # which block we are in
    
    # Compute start index for this block’s 4x4 tile
    # Each block processes 4 rows × 4 cols = 16 elements
    start_idx = bx * rows_per_block * cols_per_block

    # Load this block’s tile into shared memory
    if tid < rows_per_block * cols_per_block:
        sdata[tid] = mat[start_idx + tid]
    else:
        sdata[tid] = 0.0
    cuda.syncthreads()

    # ---- Step 1: Row-wise reduction ----
    stride_rows = rows_per_block // 2
    while stride_rows > 0:
        if tid < stride_rows * cols_per_block:
            sdata[tid] += sdata[tid + stride_rows * cols_per_block]
        stride_rows //= 2
        cuda.syncthreads()

    # ---- Step 2: Column-wise reduction ----
    stride_cols = cols_per_block // 2
    while stride_cols > 0:
        if tid < stride_cols:
            sdata[tid] += sdata[tid + stride_cols]
        stride_cols //= 2
        cuda.syncthreads()

    # ---- Step 3: Write block sum to global memory ----
    if tid == 0:
        block_sums[bx] = sdata[0]


# -------------------------------
# Host (CPU) Code
# -------------------------------

# Define a 4x16 matrix
A = np.arange(64, dtype=np.float32).reshape(4, 16)
print("Input Matrix A:\n", A)

# Flatten for GPU
A_flat = A.flatten()

# Allocate array for 4 block partial sums
block_sums = np.zeros(4, dtype=np.float32)

# Each block processes one 4x4 tile → 16 elements
threads_per_block = 16
blocks_per_grid = 4

# Launch kernel
block_tile_sum_loop[blocks_per_grid, threads_per_block](A_flat, block_sums, 4, 4)

# Compute final sum on CPU
total_sum = np.sum(block_sums)

print("\nPartial sums from each 4x4 block:", block_sums)
print("\nTotal sum of all elements in A:", total_sum)


Input Matrix A:
 [[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
 [16. 17. 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31.]
 [32. 33. 34. 35. 36. 37. 38. 39. 40. 41. 42. 43. 44. 45. 46. 47.]
 [48. 49. 50. 51. 52. 53. 54. 55. 56. 57. 58. 59. 60. 61. 62. 63.]]

Partial sums from each 4x4 block: [120. 376. 632. 888.]

Total sum of all elements in A: 2016.0


In [ ]:
# Matrix multiplication demo

In [11]:
import numpy as np
import cupy as cp
import time

# -------------------------------
# Configuration
# -------------------------------
N = 2048  # matrix size N x N (try 1024, 2048, 4096 for comparison)
print(f"Benchmarking {N}x{N} matrix multiplication")

# -------------------------------
# CPU (NumPy)
# -------------------------------
A_cpu = np.random.rand(N, N).astype(np.float32)
B_cpu = np.random.rand(N, N).astype(np.float32)

start = time.time()
C_cpu = np.dot(A_cpu, B_cpu)
cpu_time = time.time() - start

print(f"\n[CPU] Time taken: {cpu_time:.4f} seconds")

# -------------------------------
# GPU (CuPy)
# -------------------------------
A_gpu = cp.asarray(A_cpu)
B_gpu = cp.asarray(B_cpu)

cp.cuda.Device(0).synchronize()
start = time.time()
C_gpu = cp.dot(A_gpu, B_gpu)
cp.cuda.Device(0).synchronize()
gpu_time = time.time() - start

print(f"[GPU] Time taken: {gpu_time:.4f} seconds")

# -------------------------------
# Validate correctness
# -------------------------------
C_from_gpu = cp.asnumpy(C_gpu)
diff = np.mean(np.abs(C_cpu - C_from_gpu))
print(f"\nMean absolute difference between CPU & GPU results: {diff:.6e}")

# -------------------------------
# Speedup
# -------------------------------
speedup = cpu_time / gpu_time
print(f"\n🚀 GPU Speedup over CPU: {speedup:.2f}x")


Benchmarking 2048x2048 matrix multiplication

[CPU] Time taken: 0.0420 seconds
[GPU] Time taken: 0.0008 seconds

Mean absolute difference between CPU & GPU results: 7.247171e-05

🚀 GPU Speedup over CPU: 51.31x


In [8]:
! pip install cupy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 13.6 MB/s eta 0:00:00m eta 0:00:010:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached fastrlock-0.8.3-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_28_x86_64.whl.metadata (7.7 kB)
Using cached fastrlock-0.8.3-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_28_x86_64.whl (53 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for cupy (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [129 lines of output]
      Clearing directory: /tmp/pip-install-ulr4e3qx/cupy_71b2789be0094dadb84cdf80d3fb44c5/cupy/.data
      Generating CUPY_CACHE_KEY from header files...
      CUPY_CACHE_KEY (1717 files matching /tmp/pip-install-ulr4e3qx/cupy_71b2789be0094dadb84cdf80d3fb44c5/cupy/_core/include/**): 3b917dd1428e7d3cdebf5768775abe84f10b0153
      
      -------- Configuring Module: cu

In [9]:
! pip install cupy-cuda12x

  Using cached fastrlock-0.8.3-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_28_x86_64.whl.metadata (7.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.9/112.9 MB 13.7 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached fastrlock-0.8.3-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_28_x86_64.whl (53 kB)
